# Copa 2026 — Extração incremental **sem proxy**

Este notebook foi ajustado para **não usar nenhuma coluna proxy, estimada ou qualitativa** do repositório.

Ele roda dentro do repositório, lê `data/database/players_database.csv`, `matches.csv`, `team_strengths.csv` e `teams_tactical.csv` quando existirem, mas aplica uma regra rígida:

> Se o dado for proxy, estimado, qualitativo, inferido ou sem fonte direta, ele vira `NA`.

O notebook também salva os arquivos finais em:

`data/database/enriched_jogadores_tecnicos/`

E atualiza de forma incremental: quando rodar novamente, preserva valores já preenchidos e preenche apenas campos vazios/`NA`.

In [ ]:
import sys
import subprocess
import importlib.util

def ensure_package(package, import_name=None):
    import_name = import_name or package
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
    print(f"{package}: ok")

for pkg, imp in [("pandas", "pandas"), ("requests", "requests"), ("tqdm", "tqdm")]:
    ensure_package(pkg, imp)

In [ ]:
from pathlib import Path
import os
import re
import json
import time
import random
import zipfile
import hashlib
import datetime as dt
from urllib.parse import quote

import pandas as pd
import requests
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 200)

NA = "NA"

def now_iso():
    return dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat()

# ────────────────────────────────────────────
# CONFIGURAÇÃO
# ────────────────────────────────────────────

# Modo incremental: só preenche campos vazios/NA; não sobrescreve dados já preenchidos.
UPDATE_ONLY_MISSING = True

# Se quiser usar um ZIP específico, informe o caminho aqui.
# Exemplo: MANUAL_ZIP_PATH = Path("tabela-copa-main (1).zip")
MANUAL_ZIP_PATH = None

# Intervalo entre requisições reais, caso fontes públicas sejam ativadas.
REQUEST_DELAY_SECONDS = (2.5, 6.0)
REQUEST_TIMEOUT = 30

# Para teste rápido, use 10. Para processar tudo, use None.
PLAYER_LIMIT = None

# Fontes sem key. O notebook não usa proxy mesmo quando elas falham.
USE_WIKIDATA_METADATA = True
USE_WIKIPEDIA_TEXT = False

HEADERS = {
    "User-Agent": "copa-2026-sem-proxy/1.0 (educational; no api key; polite requests)"
}

def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start] + list(start.parents):
        if (p / ".git").exists() or (p / "data" / "database").exists():
            return p
    return start

REPO_ROOT = find_repo_root()
REPO_DATA_DIR = REPO_ROOT / "data" / "database"
OUTPUT_DIR = REPO_ROOT / "data" / "database" / "enriched_jogadores_tecnicos"
INPUT_DIR = REPO_ROOT / ".copa_2026_input_unzipped"
CACHE_DIR = REPO_ROOT / ".copa_2026_request_cache"

for p in [OUTPUT_DIR, INPUT_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("REPO_DATA_DIR:", REPO_DATA_DIR if REPO_DATA_DIR.exists() else "não encontrado")
print("OUTPUT_DIR:", OUTPUT_DIR)
print("UPDATE_ONLY_MISSING:", UPDATE_ONLY_MISSING)

In [ ]:
# ────────────────────────────────────────────
# Regra anti-proxy estrita
# ────────────────────────────────────────────

PROXY_FIELD_PATTERNS = [
    "proxy",
    "score",
    "forca_modelo",
    "força_modelo",
    "intensidade_valor",
    "posse_valor",
    "pressao_valor",
    "pressão_valor",
    "risco_tatico",
    "risco tático",
    "estilo",
    "encaixe",
    "observacao desempenho",
    "observação desempenho",
    "observacao",
    "observação",
    "status desempenho",
]

QUALITATIVE_OR_ESTIMATED_FIELDS = {
    "estilo_tecnico",
    "estilo técnico",
    "sistema_base",
    "sistema base",
    "intensidade",
    "posse",
    "pressao",
    "pressão",
    "risco_tatico",
    "risco tático",
    "media_proxy",
    "top11_proxy",
    "top18_proxy",
    "ataque_score",
    "meio_score",
    "defesa_score",
    "goleiro_score",
    "experiencia_score",
    "score liga/clube",
    "índice proxy 0-10",
    "indice proxy 0-10",
    "status desempenho/proxy",
    "observação desempenho",
    "observacao desempenho",
    "encaixe tático provável",
    "encaixe tatico provavel",
}

def normalize_colname(c):
    c = str(c).strip().lower()
    c = re.sub(r"[áàãâä]", "a", c)
    c = re.sub(r"[éèêë]", "e", c)
    c = re.sub(r"[íìîï]", "i", c)
    c = re.sub(r"[óòõôö]", "o", c)
    c = re.sub(r"[úùûü]", "u", c)
    c = re.sub(r"[ç]", "c", c)
    c = re.sub(r"[^a-z0-9]+", "_", c)
    return re.sub(r"_+", "_", c).strip("_")

def is_proxy_or_qualitative_column(col):
    raw = str(col).lower().strip()
    norm = normalize_colname(col)
    if raw in QUALITATIVE_OR_ESTIMATED_FIELDS or norm in {normalize_colname(x) for x in QUALITATIVE_OR_ESTIMATED_FIELDS}:
        return True
    return any(pat in raw or normalize_colname(pat) in norm for pat in PROXY_FIELD_PATTERNS)

def remove_proxy_columns(df, context="dataframe"):
    if df is None:
        return None, []
    proxy_cols = [c for c in df.columns if is_proxy_or_qualitative_column(c)]
    clean = df.drop(columns=proxy_cols, errors="ignore").copy()
    if proxy_cols:
        print(f"[ANTI-PROXY] {context}: removidas/ignoradas {len(proxy_cols)} colunas:", proxy_cols)
    return clean, proxy_cols

def mark_na_reason(reason):
    return f"NA por regra anti-proxy: {reason}"

In [ ]:
# ────────────────────────────────────────────
# HTTP com delay + cache + fontes
# ────────────────────────────────────────────

request_log = []
sources_rows = []
source_counter = 0

def safe_filename_from_url(url):
    return hashlib.sha256(str(url).encode("utf-8")).hexdigest()

def cache_path_for(url, suffix="json"):
    return CACHE_DIR / f"{safe_filename_from_url(url)}.{suffix}"

def add_source(category, entity_type, entity_name, url, publisher, title="", date_published=NA,
               data_collected="", reliability_0_100=70, notes=""):
    global source_counter
    source_counter += 1
    source_id = f"SRC{source_counter:05d}"
    sources_rows.append({
        "source_id": source_id,
        "category": category,
        "entity_type": entity_type,
        "entity_name": entity_name,
        "url": url,
        "publisher": publisher,
        "title": title,
        "date_published": date_published,
        "date_accessed": now_iso(),
        "data_collected": data_collected,
        "reliability_0_100": reliability_0_100,
        "notes": notes,
    })
    return source_id

def polite_get(url, *, as_json=False, as_bytes=False, use_cache=True, publisher="unknown", retries=2):
    suffix = "json" if as_json else "bin" if as_bytes else "txt"
    cp = cache_path_for(url, suffix=suffix)

    if use_cache and cp.exists():
        request_log.append({"url": url, "status": "cache_hit", "timestamp": now_iso()})
        if as_json:
            return json.loads(cp.read_text(encoding="utf-8"))
        if as_bytes:
            return cp.read_bytes()
        return cp.read_text(encoding="utf-8", errors="replace")

    sleep_s = random.uniform(*REQUEST_DELAY_SECONDS)
    print(f"Aguardando {sleep_s:.1f}s antes da requisição: {url[:120]}")
    time.sleep(sleep_s)

    last_err = None
    for attempt in range(retries + 1):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT)
            request_log.append({
                "url": url,
                "status_code": resp.status_code,
                "attempt": attempt + 1,
                "timestamp": now_iso(),
                "publisher": publisher,
            })
            if resp.status_code == 429 and attempt < retries:
                time.sleep(15 * (attempt + 1))
                continue
            resp.raise_for_status()

            if as_json:
                data = resp.json()
                cp.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
                return data
            if as_bytes:
                cp.write_bytes(resp.content)
                return resp.content

            cp.write_text(resp.text, encoding="utf-8")
            return resp.text
        except Exception as e:
            last_err = e
            request_log.append({"url": url, "status": "error", "error": repr(e), "attempt": attempt + 1, "timestamp": now_iso()})
            if attempt < retries:
                time.sleep(5 * (attempt + 1))

    print("Falha:", url, last_err)
    return None

In [ ]:
# ────────────────────────────────────────────
# Localização dos arquivos no repo/ZIP
# ────────────────────────────────────────────

def repo_find_file(filename):
    filename = filename.lower()
    search_dirs = [
        REPO_DATA_DIR,
        REPO_ROOT,
        REPO_ROOT / "data",
        REPO_ROOT / "datasets",
        REPO_ROOT / "notebooks",
        Path.cwd(),
        Path("/mnt/data"),
        Path("/content"),
    ]
    found = []
    for base in search_dirs:
        if base.exists():
            try:
                found.extend([p for p in base.rglob("*") if p.is_file() and p.name.lower() == filename])
            except Exception:
                pass
    found = sorted(set(found), key=lambda p: (0 if str(p).startswith(str(REPO_DATA_DIR)) else 1, len(str(p))))
    return found[0] if found else None

def repo_find_zip():
    if MANUAL_ZIP_PATH is not None:
        p = Path(MANUAL_ZIP_PATH).expanduser()
        if p.exists():
            return p
    patterns = ["tabela-copa-main*.zip", "pacote_dataset_copa_2026_features*.zip", "*copa*2026*.zip", "*.zip"]
    found = []
    for base in [REPO_ROOT, Path.cwd(), Path("/mnt/data"), Path("/content")]:
        if base.exists():
            for pattern in patterns:
                try:
                    found.extend(base.rglob(pattern))
                except Exception:
                    pass
    found = sorted(set([p for p in found if p.is_file() and p.suffix.lower() == ".zip"]),
                   key=lambda p: (0 if "tabela-copa" in p.name.lower() else 1, -p.stat().st_mtime))
    return found[0] if found else None

def read_csv_flexible(path):
    if path is None:
        return None
    try:
        return pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[])
    except UnicodeDecodeError:
        return pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[], encoding="latin1")

def find_file_by_name_in_input(name):
    matches = [p for p in INPUT_DIR.rglob("*") if p.is_file() and p.name.lower() == name.lower()]
    return matches[0] if matches else None

required_candidates = {
    "players_database": repo_find_file("players_database.csv"),
    "team_strengths": repo_find_file("team_strengths.csv"),
    "teams_tactical": repo_find_file("teams_tactical.csv"),
    "matches": repo_find_file("matches.csv"),
    "dataset_features": repo_find_file("dataset_copa_2026_features_pesquisadas.csv"),
    "sources_existing": repo_find_file("sources_used.csv"),
}

if required_candidates["players_database"] is None and required_candidates["dataset_features"] is None:
    zip_path = repo_find_zip()
    if zip_path is None:
        try:
            from google.colab import files
            print("Nenhum ZIP encontrado. Envie o ZIP do repositório ou pacote...")
            uploaded = files.upload()
            if uploaded:
                zip_path = Path(next(iter(uploaded.keys()))).resolve()
        except Exception:
            pass

    if zip_path is None:
        raise FileNotFoundError("Não encontrei os CSVs nem um ZIP. Coloque o repositório/ZIP na pasta do notebook.")

    print("ZIP selecionado:", zip_path)

    for old in sorted(INPUT_DIR.rglob("*"), reverse=True):
        if old.is_file():
            old.unlink()
        elif old.is_dir():
            try:
                old.rmdir()
            except OSError:
                pass

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(INPUT_DIR)

    fallback = {
        "players_database": find_file_by_name_in_input("players_database.csv"),
        "team_strengths": find_file_by_name_in_input("team_strengths.csv"),
        "teams_tactical": find_file_by_name_in_input("teams_tactical.csv"),
        "matches": find_file_by_name_in_input("matches.csv"),
        "dataset_features": find_file_by_name_in_input("dataset_copa_2026_features_pesquisadas.csv"),
        "sources_existing": find_file_by_name_in_input("sources_used.csv"),
    }
    for k, v in fallback.items():
        if required_candidates[k] is None and v is not None:
            required_candidates[k] = v

print("Arquivos usados como entrada:")
for k, v in required_candidates.items():
    print(f"{k}: {v if v else 'NÃO ENCONTRADO'}")

players_db_raw = read_csv_flexible(required_candidates["players_database"])
team_strengths_raw = read_csv_flexible(required_candidates["team_strengths"])
teams_tactical_raw = read_csv_flexible(required_candidates["teams_tactical"])
matches_df = read_csv_flexible(required_candidates["matches"])
dataset_features = read_csv_flexible(required_candidates["dataset_features"])
existing_sources = read_csv_flexible(required_candidates["sources_existing"])

# Remove colunas proxy/qualitativas para impedir uso acidental.
players_db, players_proxy_cols = remove_proxy_columns(players_db_raw, "players_database.csv")
team_strengths, strengths_proxy_cols = remove_proxy_columns(team_strengths_raw, "team_strengths.csv")
teams_tactical, tactical_proxy_cols = remove_proxy_columns(teams_tactical_raw, "teams_tactical.csv")

for name, df in [
    ("players_database_sem_proxy", players_db),
    ("team_strengths_sem_proxy", team_strengths),
    ("teams_tactical_sem_proxy", teams_tactical),
    ("matches", matches_df),
    ("dataset_features", dataset_features),
]:
    print(name, "=>", None if df is None else df.shape)

In [ ]:
# ────────────────────────────────────────────
# Padronização sem uso de proxy
# ────────────────────────────────────────────

def standardize_players_df(df):
    if df is None:
        return pd.DataFrame(columns=[
            "player_id", "selecao", "jogador", "posicao", "clube",
            "pais_do_clube", "liga", "idade", "caps_selecao", "gols_selecao",
            "player_source"
        ])

    out = df.copy()
    original_cols = list(out.columns)
    out.columns = [normalize_colname(c) for c in out.columns]

    aliases = {
        "codigo": "codigo",
        "n": "numero",
        "no": "numero",
        "nº": "numero",
        "selecao": "selecao",
        "seleção": "selecao",
        "jogador": "jogador",
        "nome": "jogador",
        "player": "jogador",
        "player_name": "jogador",
        "posicao": "posicao",
        "posição": "posicao",
        "clube": "clube",
        "club": "clube",
        "pais_do_clube": "pais_do_clube",
        "país_do_clube": "pais_do_clube",
        "idade_em_26_06_2026": "idade",
        "idade": "idade",
        "caps_selecao": "caps_selecao",
        "caps_seleção": "caps_selecao",
        "gols_selecao": "gols_selecao",
        "gols_seleção": "gols_selecao",
        "fonte_convocacao": "player_source",
        "fonte_convocação": "player_source",
    }

    normalized = pd.DataFrame()
    for old, new in aliases.items():
        if old in out.columns and new not in normalized.columns:
            normalized[new] = out[old]

    required = ["codigo", "numero", "selecao", "jogador", "posicao", "clube", "pais_do_clube", "idade", "caps_selecao", "gols_selecao", "player_source"]
    for c in required:
        if c not in normalized.columns:
            normalized[c] = NA

    normalized["player_id"] = normalized.apply(
        lambda r: f"{r['codigo']}_{r['numero']}_{r['jogador']}".replace(" ", "_") if r["codigo"] != NA else str(r["jogador"]).replace(" ", "_"),
        axis=1
    )

    normalized["liga"] = NA

    final_cols = ["player_id", "selecao", "jogador", "posicao", "clube", "pais_do_clube", "liga", "idade", "caps_selecao", "gols_selecao", "player_source"]
    return normalized[final_cols].replace({"": NA})

players_base = standardize_players_df(players_db)
if PLAYER_LIMIT is not None:
    players_base = players_base.head(PLAYER_LIMIT).copy()

def get_teams_from_inputs():
    teams = []
    if matches_df is not None:
        tmp = matches_df.copy()
        tmp.columns = [normalize_colname(c) for c in tmp.columns]
        for c in ["equipe1", "equipe2"]:
            if c in tmp.columns:
                teams += list(tmp[c].astype(str))
    if "selecao" in players_base.columns:
        teams += list(players_base["selecao"].astype(str))
    return sorted({x for x in teams if x and x != NA})

teams = get_teams_from_inputs()

print("Jogadores:", len(players_base))
print("Seleções:", len(teams))
display(players_base.head())

In [ ]:
# ────────────────────────────────────────────
# Wikidata opcional: somente metadados diretos, não desempenho/proxy
# ────────────────────────────────────────────

WIKIDATA_API = "https://www.wikidata.org/w/api.php"
WIKIDATA_ENTITY = "https://www.wikidata.org/wiki/Special:EntityData/{qid}.json"

def wikidata_search_entity(search, language="en", limit=5):
    url = f"{WIKIDATA_API}?action=wbsearchentities&search={quote(search)}&language={language}&format=json&limit={limit}"
    data = polite_get(url, as_json=True, publisher="Wikidata")
    return data.get("search", []) if data else []

def wikidata_get_entity(qid):
    data = polite_get(WIKIDATA_ENTITY.format(qid=qid), as_json=True, publisher="Wikidata")
    return data.get("entities", {}).get(qid) if data else None

def wd_claim_values(entity, prop):
    return [c.get("mainsnak", {}).get("datavalue", {}).get("value") for c in entity.get("claims", {}).get(prop, [])]

def wd_entity_id_from_value(value):
    if isinstance(value, dict) and "id" in value:
        return value["id"]
    if isinstance(value, dict) and "numeric-id" in value:
        return "Q" + str(value["numeric-id"])
    return None

def wd_label_from_entity_id(qid, lang_priority=("pt", "en", "es", "fr")):
    ent = wikidata_get_entity(qid)
    if not ent:
        return NA
    labels = ent.get("labels", {})
    for lang in lang_priority:
        if lang in labels:
            return labels[lang]["value"]
    return next(iter(labels.values()))["value"] if labels else NA

def wikidata_player_metadata(name):
    if not USE_WIKIDATA_METADATA or not name or name == NA:
        return {"source_id": NA, "notes": "Wikidata disabled"}
    candidates = wikidata_search_entity(f"{name} association football", language="en", limit=5) or wikidata_search_entity(name, language="en", limit=5)
    if not candidates:
        return {"source_id": NA, "notes": "Wikidata entity not found"}
    qid = candidates[0]["id"]
    ent = wikidata_get_entity(qid)
    if not ent:
        return {"source_id": NA, "notes": "Wikidata entity fetch failed"}

    positions, clubs = [], []
    for val in wd_claim_values(ent, "P413"):
        eid = wd_entity_id_from_value(val)
        if eid:
            positions.append(wd_label_from_entity_id(eid))
    for val in wd_claim_values(ent, "P54"):
        eid = wd_entity_id_from_value(val)
        if eid:
            clubs.append(wd_label_from_entity_id(eid))

    sid = add_source(
        category="player_stats",
        entity_type="player",
        entity_name=name,
        url=WIKIDATA_ENTITY.format(qid=qid),
        publisher="Wikidata",
        title=f"Wikidata entity {qid}",
        data_collected="direct metadata only: position/team claims when available",
        reliability_0_100=65,
        notes="No performance stats. No proxy."
    )

    return {
        "position_labels": "; ".join(positions) if positions else NA,
        "club_labels": "; ".join(clubs) if clubs else NA,
        "source_id": sid,
        "notes": "Wikidata direct metadata collected. No proxy."
    }

In [ ]:
# ────────────────────────────────────────────
# player_stats_2025_26.csv sem proxy
# ────────────────────────────────────────────

player_stats_cols = [
    "player_id","selecao","jogador","posicao","clube","pais_do_clube","liga","idade","caps_selecao","gols_selecao",
    "minutes_last_365","matches_played_last_365","matches_started_last_365","club_goals_last_365","club_assists_last_365",
    "xg_last_365","xa_last_365","shots_last_365","shots_on_target_last_365","key_passes_last_365",
    "progressive_passes_last_365","progressive_carries_last_365","tackles_last_365","interceptions_last_365",
    "blocks_last_365","clearances_last_365","aerial_duels_won_pct","pass_completion_pct",
    "yellow_cards_last_365","red_cards_last_365","avg_player_rating_last_365",
    "gk_minutes_last_365","gk_matches_last_365","gk_save_pct_last_365","gk_clean_sheets_last_365",
    "gk_goals_against_last_365","gk_xg_against_last_365","gk_goals_prevented_last_365","gk_penalty_save_pct_last_365",
    "injury_status","injury_type","expected_return_date","suspension_status","suspension_reason","fitness_status",
    "expected_starter","starter_confidence_0_100",
    "market_value_eur","club_strength_0_100","league_strength_0_100","is_top5_league","played_elite_competition",
    "elite_competition_minutes_last_365",
    "player_stats_source","injury_source","market_value_source","last_updated_at","research_confidence_0_100",
    "missing_fields_count","notes"
]

def count_missing(row, cols):
    return sum(1 for c in cols if str(row.get(c, NA)).strip() in ["", NA, "nan", "NaN", "None", "<NA>"])

player_rows = []

for _, base in tqdm(players_base.iterrows(), total=len(players_base), desc="Jogadores"):
    row = {c: NA for c in player_stats_cols}

    for c in ["player_id","selecao","jogador","posicao","clube","pais_do_clube","liga","idade","caps_selecao","gols_selecao"]:
        row[c] = base.get(c, NA) if str(base.get(c, "")).strip() else NA

    # Fonte direta da convocação/base oficial, quando existente.
    direct_source = base.get("player_source", NA)
    source_ids = []
    notes = []

    if direct_source != NA:
        sid = add_source(
            category="player_stats",
            entity_type="player",
            entity_name=row["jogador"],
            url=direct_source,
            publisher="Base/repositório",
            title="Fonte de convocação informada em players_database.csv",
            data_collected="direct squad metadata: player, position, club, age, caps, national goals",
            reliability_0_100=85,
            notes="Used only direct factual fields. Proxy columns from repository were ignored."
        )
        source_ids.append(sid)

    # Wikidata apenas completa metadados vazios, não estatísticas.
    if USE_WIKIDATA_METADATA and row["jogador"] != NA:
        wd = wikidata_player_metadata(row["jogador"])
        if row["posicao"] == NA and wd.get("position_labels", NA) != NA:
            row["posicao"] = wd["position_labels"]
        if row["clube"] == NA and wd.get("club_labels", NA) != NA:
            row["clube"] = wd["club_labels"]
        if wd.get("source_id", NA) != NA:
            source_ids.append(wd["source_id"])
        notes.append(wd.get("notes", ""))

    # Campos que seriam proxy/estimativa ficam NA.
    row["is_top5_league"] = NA
    row["expected_starter"] = "unknown"
    row["player_stats_source"] = ";".join(source_ids) if source_ids else NA
    row["injury_source"] = NA
    row["market_value_source"] = NA
    row["last_updated_at"] = now_iso()
    row["research_confidence_0_100"] = 70 if source_ids else 20
    notes.append("Proxy proibido: desempenho de clube, xG/xA, rating, lesões, suspensões, valor de mercado, força de clube/liga e titularidade ficam NA sem fonte direta.")
    row["notes"] = " | ".join([n for n in notes if n])
    row["missing_fields_count"] = count_missing(row, player_stats_cols)

    player_rows.append(row)

player_stats_2025_26 = pd.DataFrame(player_rows, columns=player_stats_cols)
display(player_stats_2025_26.head())

In [ ]:
# ────────────────────────────────────────────
# Agregação por seleção: somente soma/média de campos diretos disponíveis
# Sem top18/starting11 estimados
# ────────────────────────────────────────────

team_agg_cols = [
    "selecao","squad_players_count","squad_avg_age","squad_total_caps","squad_total_international_goals",
    "squad_minutes_last_365","squad_goals_club_last_365","squad_assists_club_last_365","squad_xg_last_365","squad_xa_last_365",
    "squad_avg_player_rating_last_365","squad_avg_market_value_eur","squad_total_market_value_eur","squad_pct_top5_leagues",
    "squad_avg_league_strength_0_100","squad_avg_club_strength_0_100","squad_elite_competition_minutes_last_365",
    "top18_avg_rating_last_365","top18_minutes_last_365","top18_goals_club_last_365","top18_assists_club_last_365",
    "top18_xg_last_365","top18_xa_last_365","top18_total_market_value_eur","top18_pct_top5_leagues",
    "starting11_avg_age","starting11_total_caps","starting11_total_international_goals","starting11_minutes_last_365",
    "starting11_goals_club_last_365","starting11_assists_club_last_365","starting11_xg_last_365","starting11_xa_last_365",
    "starting11_avg_rating_last_365","starting11_total_market_value_eur","starting11_pct_top5_leagues",
    "starting11_avg_league_strength_0_100","starting11_avg_club_strength_0_100",
    "expected_starters_available","key_injuries_count","suspended_players_count","doubtful_players_count",
    "rotation_risk_0_100","lineup_source_type","last_updated_at","missing_fields_count","notes"
]

def to_num(series):
    return pd.to_numeric(series.replace({NA: pd.NA, "": pd.NA}), errors="coerce")

def safe_sum(series):
    nums = to_num(series)
    return NA if nums.notna().sum() == 0 else float(nums.sum())

def safe_mean(series):
    nums = to_num(series)
    return NA if nums.notna().sum() == 0 else float(nums.mean())

team_rows = []
for team in teams:
    g = player_stats_2025_26[player_stats_2025_26["selecao"] == team]
    row = {c: NA for c in team_agg_cols}
    row["selecao"] = team
    row["squad_players_count"] = int(len(g)) if len(g) else NA
    row["squad_avg_age"] = safe_mean(g["idade"]) if len(g) else NA
    row["squad_total_caps"] = safe_sum(g["caps_selecao"]) if len(g) else NA
    row["squad_total_international_goals"] = safe_sum(g["gols_selecao"]) if len(g) else NA
    row["lineup_source_type"] = "unknown"
    row["last_updated_at"] = now_iso()
    row["notes"] = "Sem proxy: top18, starting11, rotação, injuries/suspensions e métricas de desempenho ficam NA sem fonte direta."
    row["missing_fields_count"] = count_missing(row, team_agg_cols)
    team_rows.append(row)

team_player_aggregates = pd.DataFrame(team_rows, columns=team_agg_cols)
display(team_player_aggregates.head())

In [ ]:
# ────────────────────────────────────────────
# Tático, técnicos, lineups: sem proxy
# ────────────────────────────────────────────

team_tactical_cols = [
    "selecao","coach_name","main_formation","formation_last_match","formation_second_option","matches_analyzed_count",
    "avg_possession_last_10","shots_for_last_10","shots_against_last_10","shots_on_target_for_last_10","shots_on_target_against_last_10",
    "goals_for_last_10","goals_against_last_10","xg_for_last_10","xg_against_last_10","corners_for_last_10","corners_against_last_10",
    "passes_attempted_last_10","pass_accuracy_pct_last_10","long_ball_pct_last_10","crosses_last_10","counter_attack_goals_last_10",
    "set_piece_goals_for_last_10","set_piece_goals_against_last_10",
    "ppda","high_turnovers_last_10","recoveries_final_third_last_10","pressing_intensity_0_100","defensive_block_height_0_100",
    "directness_0_100","tempo_0_100","risk_tolerance_0_100","width_0_100","set_piece_attack_0_100","set_piece_defense_0_100",
    "counter_attack_style_0_100","possession_style_0_100",
    "tactical_data_type","tactical_source","last_updated_at","research_confidence_0_100","missing_fields_count","notes"
]

team_tactical_real_stats = pd.DataFrame([
    {
        **{c: NA for c in team_tactical_cols},
        "selecao": team,
        "tactical_data_type": "unavailable",
        "last_updated_at": now_iso(),
        "research_confidence_0_100": 0,
        "notes": "Sem proxy: colunas qualitativas/estimadas do repositório foram ignoradas. Métricas táticas ficam NA sem fonte direta."
    }
    for team in teams
], columns=team_tactical_cols)
team_tactical_real_stats["missing_fields_count"] = team_tactical_real_stats.apply(lambda r: count_missing(r, team_tactical_cols), axis=1)

team_coach_cols = [
    "selecao","coach_name","coach_nationality","coach_start_date","coach_days_in_charge",
    "coach_matches","coach_wins","coach_draws","coach_losses","coach_win_rate",
    "coach_goals_for","coach_goals_against","coach_goals_for_avg","coach_goals_against_avg",
    "coach_points_per_match","coach_major_tournaments","coach_main_formation",
    "coach_style_description","coach_style_source","coach_stats_source",
    "last_updated_at","research_confidence_0_100","missing_fields_count","notes"
]

# Técnico/nacionalidade são campos factuais do repositório. Estilo/sistema são ignorados por serem qualitativos.
coach_rows = []
if players_db_raw is not None:
    tmp = players_db_raw.copy()
    tmp.columns = [normalize_colname(c) for c in tmp.columns]
else:
    tmp = pd.DataFrame()

for team in teams:
    row = {c: NA for c in team_coach_cols}
    row["selecao"] = team
    if not tmp.empty and "selecao" in tmp.columns:
        g = tmp[tmp["selecao"].astype(str) == team]
        if len(g):
            if "tecnico" in g.columns:
                vals = [x for x in g["tecnico"].dropna().astype(str).unique() if x and x != NA]
                if vals:
                    row["coach_name"] = vals[0]
            if "nacionalidade_tecnico" in g.columns:
                vals = [x for x in g["nacionalidade_tecnico"].dropna().astype(str).unique() if x and x != NA]
                if vals:
                    row["coach_nationality"] = vals[0]
    row["last_updated_at"] = now_iso()
    row["research_confidence_0_100"] = 65 if row["coach_name"] != NA else 0
    row["notes"] = "Sem proxy: estilo, sistema, aproveitamento e formações ficam NA sem fonte direta estruturada."
    row["missing_fields_count"] = count_missing(row, team_coach_cols)
    coach_rows.append(row)

team_coach_real_stats = pd.DataFrame(coach_rows, columns=team_coach_cols)

lineups_cols = [
    "jogo","data","equipe","adversario","fase","grupo",
    "expected_lineup_gk","expected_lineup_defenders","expected_lineup_midfielders","expected_lineup_forwards",
    "expected_formation","expected_starters_count","expected_starters_available",
    "key_injuries_count","key_injured_players","doubtful_players_count","doubtful_players",
    "suspended_players_count","suspended_players","rotation_risk_0_100",
    "lineup_source_type","lineup_source","injury_source","suspension_source",
    "last_updated_at","research_confidence_0_100","missing_fields_count","notes"
]

lineup_rows = []
base_matches = matches_df
if base_matches is None and dataset_features is not None:
    base_matches = dataset_features

if base_matches is not None:
    m = base_matches.copy()
    m.columns = [normalize_colname(c) for c in m.columns]
    for c in ["jogo", "data", "fase", "grupo", "equipe1", "equipe2"]:
        if c not in m.columns:
            m[c] = NA
    for _, r in m[["jogo", "data", "fase", "grupo", "equipe1", "equipe2"]].drop_duplicates().iterrows():
        for team_col, opp_col in [("equipe1", "equipe2"), ("equipe2", "equipe1")]:
            team = r[team_col]
            if str(team).strip() and team != NA:
                row = {c: NA for c in lineups_cols}
                row.update({
                    "jogo": r["jogo"],
                    "data": r["data"],
                    "equipe": team,
                    "adversario": r[opp_col],
                    "fase": r["fase"],
                    "grupo": r["grupo"],
                    "lineup_source_type": "unknown",
                    "last_updated_at": now_iso(),
                    "research_confidence_0_100": 0,
                    "notes": "Sem proxy: escalações, lesões, suspensões e rotação ficam NA sem fonte direta pré-jogo."
                })
                row["missing_fields_count"] = count_missing(row, lineups_cols)
                lineup_rows.append(row)

lineups_injuries_suspensions = pd.DataFrame(lineup_rows, columns=lineups_cols)

display(team_tactical_real_stats.head(), team_coach_real_stats.head(), lineups_injuries_suspensions.head())

In [ ]:
# ────────────────────────────────────────────
# Dataset final sem proxy
# ────────────────────────────────────────────

final_feature_cols = [
    "starting11_avg_rating_last_365","starting11_minutes_last_365","starting11_goals_club_last_365",
    "starting11_assists_club_last_365","starting11_xg_last_365","starting11_xa_last_365",
    "key_injuries_count","suspended_players_count","expected_starters_available",
    "avg_possession_last_10","shots_for_last_10","shots_against_last_10",
    "pressing_intensity_0_100","directness_0_100","defensive_block_height_0_100",
    "coach_days_in_charge","coach_win_rate","coach_goals_for_avg","coach_goals_against_avg"
]

if dataset_features is not None:
    final = dataset_features.copy()
    final.columns = [normalize_colname(c) for c in final.columns]
else:
    final = matches_df.copy() if matches_df is not None else pd.DataFrame(columns=["jogo", "data", "fase", "grupo", "equipe1", "equipe2"])
    final.columns = [normalize_colname(c) for c in final.columns]

for c in ["equipe1", "equipe2"]:
    if c not in final.columns:
        final[c] = NA

agg = team_player_aggregates.set_index("selecao")
tact = team_tactical_real_stats.set_index("selecao")
coach = team_coach_real_stats.set_index("selecao")

for side, team_col in [("team1", "equipe1"), ("team2", "equipe2")]:
    for feat in final_feature_cols:
        vals = []
        for team in final[team_col].astype(str):
            v = NA
            if feat in agg.columns and team in agg.index:
                v = agg.loc[team, feat]
            elif feat in tact.columns and team in tact.index:
                v = tact.loc[team, feat]
            elif feat in coach.columns and team in coach.index:
                v = coach.loc[team, feat]
            vals.append(v)
        final[f"{side}_{feat}"] = vals

# Diferenças/somas são operações diretas, não proxy, mas só geram valor se ambos os lados tiverem dado real.
for feat in final_feature_cols:
    t1 = pd.to_numeric(final[f"team1_{feat}"].replace({NA: pd.NA, "": pd.NA}), errors="coerce")
    t2 = pd.to_numeric(final[f"team2_{feat}"].replace({NA: pd.NA, "": pd.NA}), errors="coerce")
    final[f"diff_{feat}"] = (t1 - t2).astype("Float64").astype(str).replace({"<NA>": NA})
    final[f"sum_{feat}"] = (t1 + t2).astype("Float64").astype(str).replace({"<NA>": NA})

# Esses campos seriam confrontos/proxies. Mantém NA.
for c in [
    "style_similarity_0_100",
    "team1_press_vs_team2_possession",
    "team2_press_vs_team1_possession",
    "team1_direct_attack_vs_team2_block",
    "team2_direct_attack_vs_team1_block",
]:
    final[c] = NA

final["data_leakage_risk_players"] = "uncertain_check_source"
final["data_leakage_risk_tactical"] = "uncertain_check_source"
final["data_leakage_risk_lineups"] = "uncertain_check_source"
final["data_leakage_risk_coach"] = "uncertain_check_source"

dataset_copa_2026_features_jogadores_tecnicos = final
display(dataset_copa_2026_features_jogadores_tecnicos.head())

In [ ]:
# ────────────────────────────────────────────
# Dicionário de dados
# ────────────────────────────────────────────

def dictionary_for(df, category, notes):
    rows = []
    for c in df.columns:
        rows.append({
            "column_name": c,
            "description": c.replace("_", " "),
            "data_type": "string",
            "unit": "NA",
            "allowed_values": "NA",
            "source_category": category,
            "pre_match_allowed": "yes",
            "data_leakage_risk": "uncertain_check_source",
            "calculation_method": "direct_from_source_or_NA_no_proxy",
            "notes": notes
        })
    return rows

dict_rows = []
dict_rows += dictionary_for(player_stats_2025_26, "player_stats", "Sem proxy. Campos sem fonte direta ficam NA.")
dict_rows += dictionary_for(team_tactical_real_stats, "team_tactical", "Sem proxy. Métricas táticas ficam NA sem fonte direta.")
dict_rows += dictionary_for(team_coach_real_stats, "coach_stats", "Sem proxy. Campos qualitativos foram ignorados.")
dict_rows += dictionary_for(lineups_injuries_suspensions, "lineup", "Sem proxy. Lineups/lesões/suspensões ficam NA sem fonte direta.")
dict_rows += dictionary_for(dataset_copa_2026_features_jogadores_tecnicos, "final_dataset", "Sem proxy. Operações diff/sum só usam valores diretos disponíveis.")

data_dictionary_jogadores_tecnicos = pd.DataFrame(dict_rows).drop_duplicates(subset=["column_name", "source_category"])
display(data_dictionary_jogadores_tecnicos.head())

In [ ]:
# ────────────────────────────────────────────
# Salvamento incremental
# ────────────────────────────────────────────

def clean_for_csv(df):
    return df.copy().replace({pd.NA: NA, None: NA, "nan": NA, "NaN": NA}).fillna(NA)

def is_missing_value(x):
    return pd.isna(x) or str(x).strip() in ["", NA, "nan", "NaN", "None", "<NA>"]

def merge_incremental(existing_df, new_df, key_cols, update_only_missing=True):
    if existing_df is None or existing_df.empty:
        return new_df.copy()

    old = existing_df.copy()
    new = new_df.copy()

    for c in new.columns:
        if c not in old.columns:
            old[c] = NA
    for c in old.columns:
        if c not in new.columns:
            new[c] = NA

    cols = list(dict.fromkeys(list(old.columns) + list(new.columns)))
    old = old[cols]
    new = new[cols]

    old["_key"] = old[key_cols].astype(str).agg("||".join, axis=1)
    new["_key"] = new[key_cols].astype(str).agg("||".join, axis=1)

    rows = old.to_dict("records")
    idx = {r["_key"]: i for i, r in enumerate(rows)}

    for _, nr in new.iterrows():
        d = nr.to_dict()
        k = d["_key"]
        if k not in idx:
            rows.append(d)
            continue

        r = rows[idx[k]]
        for c in cols:
            if c == "_key":
                continue
            new_val = d.get(c, NA)
            old_val = r.get(c, NA)

            if update_only_missing:
                if is_missing_value(old_val) and not is_missing_value(new_val):
                    r[c] = new_val
            else:
                if not is_missing_value(new_val):
                    r[c] = new_val

    return pd.DataFrame(rows).drop(columns=["_key"], errors="ignore")[cols]

def save_csv_incremental(df, filename, key_cols):
    path = OUTPUT_DIR / filename
    new_df = clean_for_csv(df)

    if path.exists():
        old_df = pd.read_csv(path, dtype=str, keep_default_na=False, na_values=[])
        merged = merge_incremental(old_df, new_df, key_cols, UPDATE_ONLY_MISSING)
    else:
        merged = new_df

    merged.to_csv(path, index=False, encoding="utf-8", na_rep=NA)
    return path, merged

sources_used_jogadores_tecnicos = pd.DataFrame(sources_rows, columns=[
    "source_id","category","entity_type","entity_name","url","publisher","title",
    "date_published","date_accessed","data_collected","reliability_0_100","notes"
])

if sources_used_jogadores_tecnicos.empty:
    sources_used_jogadores_tecnicos = pd.DataFrame([{
        "source_id": NA,
        "category": NA,
        "entity_type": NA,
        "entity_name": NA,
        "url": NA,
        "publisher": NA,
        "title": NA,
        "date_published": NA,
        "date_accessed": now_iso(),
        "data_collected": "No source accessed",
        "reliability_0_100": 0,
        "notes": "Sem fonte externa usada. Sem proxy."
    }])

request_log_df = pd.DataFrame(request_log) if request_log else pd.DataFrame([{"timestamp": now_iso(), "status": "no_requests"}])

saved = {}
saved["player_stats_2025_26.csv"] = save_csv_incremental(player_stats_2025_26, "player_stats_2025_26.csv", ["player_id", "selecao", "jogador"])
saved["team_tactical_real_stats.csv"] = save_csv_incremental(team_tactical_real_stats, "team_tactical_real_stats.csv", ["selecao"])
saved["team_coach_real_stats.csv"] = save_csv_incremental(team_coach_real_stats, "team_coach_real_stats.csv", ["selecao", "coach_name"])
saved["lineups_injuries_suspensions.csv"] = save_csv_incremental(lineups_injuries_suspensions, "lineups_injuries_suspensions.csv", ["jogo", "equipe", "adversario"])
saved["dataset_copa_2026_features_jogadores_tecnicos.csv"] = save_csv_incremental(
    dataset_copa_2026_features_jogadores_tecnicos,
    "dataset_copa_2026_features_jogadores_tecnicos.csv",
    ["jogo"] if "jogo" in dataset_copa_2026_features_jogadores_tecnicos.columns else ["equipe1", "equipe2"]
)
saved["data_dictionary_jogadores_tecnicos.csv"] = save_csv_incremental(data_dictionary_jogadores_tecnicos, "data_dictionary_jogadores_tecnicos.csv", ["column_name", "source_category"])
saved["sources_used_jogadores_tecnicos.csv"] = save_csv_incremental(sources_used_jogadores_tecnicos, "sources_used_jogadores_tecnicos.csv", ["category", "entity_type", "entity_name", "url"])
request_log_path, request_log_final = save_csv_incremental(request_log_df, "request_log.csv", ["timestamp", "status"])

report_lines = [
    "# Relatório incremental sem proxy — jogadores, técnicos e seleções",
    "",
    f"Coleta gerada em: {now_iso()}",
    f"Repositório: {REPO_ROOT}",
    f"Saída: {OUTPUT_DIR}",
    f"UPDATE_ONLY_MISSING: {UPDATE_ONLY_MISSING}",
    "",
    "## Regra aplicada",
    "Não foram usados proxies, estimativas, rankings derivados, scores qualitativos, estilo inferido ou titularidade estimada.",
    "Colunas do repositório com 'proxy', 'score', 'estilo', 'intensidade', 'pressão', 'risco tático', 'encaixe' ou observações qualitativas foram ignoradas.",
    "",
    "## Colunas ignoradas por anti-proxy",
    f"- players_database.csv: {players_proxy_cols}",
    f"- team_strengths.csv: {strengths_proxy_cols}",
    f"- teams_tactical.csv: {tactical_proxy_cols}",
    "",
    "## Arquivos salvos",
]

for filename, (path, df) in saved.items():
    report_lines.append(f"- {filename}: {df.shape[0]} linhas x {df.shape[1]} colunas")

report_path = OUTPUT_DIR / "relatorio_pesquisa_jogadores_tecnicos.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")

zip_final_path = OUTPUT_DIR / "pacote_jogadores_tecnicos_copa_2026.zip"
with zipfile.ZipFile(zip_final_path, "w", zipfile.ZIP_DEFLATED) as z:
    for filename, (path, df) in saved.items():
        z.write(path, arcname=filename)
    z.write(report_path, arcname=report_path.name)
    z.write(request_log_path, arcname=request_log_path.name)

print("Salvo em:", OUTPUT_DIR)
print("ZIP final:", zip_final_path)
for filename, (path, df) in saved.items():
    print("-", filename, df.shape)

## Como atualizar depois

Rode o notebook novamente dentro do mesmo repositório.

Com:

```python
UPDATE_ONLY_MISSING = True
```

ele só preenche campos vazios/`NA`.  
Ele **não** usa proxy e **não** transforma campos ausentes em estimativa.

Se em uma próxima versão você adicionar uma fonte direta e confiável para algum campo, o notebook preencherá apenas os campos ainda ausentes.